# Bab 4 - Latihan: Ekstraktor Laporan Insiden

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/dteti-workshop/workshop-genai-migas/blob/main/colab/hari3/04-keluaran-terstruktur-agen-latihan-1.ipynb)

Ini versi **Google Colab**. Isi dan urutan contohnya sama dengan versi VS Code; yang berbeda hanya cell Persiapan di bawah, yang menyalin repository, memasang pustaka, dan mengisi API key di runtime Colab.

1. Pilih **File > Save a copy in Drive** agar perubahan tersimpan di akun Anda sendiri.
2. Jalankan tiga cell **Persiapan Colab** berurutan (sekitar 2-3 menit pada sesi baru).
3. Kerjakan dari atas ke bawah dengan `Shift+Enter`.

Kode jawaban latihan ada di bagian **Kode Jawaban Latihan** pada [PDF bab ini](https://github.com/dteti-workshop/workshop-genai-migas/blob/main/hari3/04-keluaran-terstruktur-agen/04-keluaran-terstruktur-agen.pdf). Salin kodenya ke cell yang bertanda `TODO`.

Kalau runtime terputus, file di `/content` (termasuk `.env`) hilang. Jalankan ulang Persiapan, lalu lanjutkan dari cell yang sedang Anda kerjakan.

## Persiapan Colab
Jalankan tiga cell berikut berurutan sebelum cell lain di notebook ini.

In [ ]:
# Persiapan 1 - Menyalin repository dan memasang pustaka (sekitar 2-3 menit pada sesi baru)
import os
import subprocess
import sys

REPO_URL = "https://github.com/dteti-workshop/workshop-genai-migas.git"
REPO = "/content/workshop-genai-migas" if os.path.isdir("/content") else os.path.abspath("workshop-genai-migas")
if not os.path.isdir(REPO):
    subprocess.run(["git", "clone", "-q", REPO_URL, REPO], check=True)
os.chdir(REPO)

pasang = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "colab/requirements-colab.txt", "-e", "."],
                        capture_output=True, text=True)
if pasang.returncode != 0:
    print(pasang.stderr[-3000:])
    raise RuntimeError("Instalasi pustaka gagal. Jalankan ulang cell ini.")

# "-e ." memasang common dan migas_shared agar subproses (server MCP) dapat mengimpornya; kernel yang sedang berjalan
# belum memuat instalasi baru, jadi folder repo juga ditambahkan ke sys.path
if REPO not in sys.path:
    sys.path.insert(0, REPO)
os.environ["PYTHONPATH"] = REPO + os.pathsep + os.environ.get("PYTHONPATH", "")
print("Repository dan pustaka siap. Folder kerja:", os.getcwd())

In [ ]:
# Persiapan 2 - Mengisi ID peserta dan API key (dilewati bila .env sudah berisi key)
import re
from getpass import getpass
from pathlib import Path

from dotenv import load_dotenv

env = Path(REPO) / ".env"


def rahasia(nama: str):
    """Nilai dari Colab Secrets (ikon kunci di panel kiri), atau None bila tidak ada."""
    try:
        from google.colab import userdata
        return userdata.get(nama)
    except Exception:  # noqa: BLE001 - bukan di Colab, secret belum dibuat, atau akses notebook belum diizinkan
        return None


if not (env.exists() and re.search(r"(?m)^ONLINE_API_KEY=(?!isi-dengan)\S+", env.read_text(encoding="utf-8"))):
    peserta = (rahasia("PARTICIPANT_ID") or input("ID peserta (nomor meja dan nama singkat, contoh P07-andi): ")).strip()
    kunci = (rahasia("ONLINE_API_KEY") or getpass("API key workshop (ketikan tidak ditampilkan): ")).strip()
    if not kunci:
        raise ValueError("API key belum diisi. Jalankan ulang cell ini.")
    isi = (Path(REPO) / ".env.example").read_text(encoding="utf-8")
    for nama, nilai in (("PARTICIPANT_ID", peserta or "P00-nama"), ("ONLINE_API_KEY", kunci)):
        isi = re.sub(rf"(?m)^{nama}=.*$", lambda _, n=nama, v=nilai: f"{n}={v}", isi)
    env.write_text(isi, encoding="utf-8")
    del kunci

# Nilai dari .env dimuat ulang; modul yang sudah diimpor dibuang agar membaca nilai terbaru
load_dotenv(env, override=True)
for modul in [m for m in sys.modules if m.split(".")[0] in ("common", "migas_shared", "rag_pipeline")]:
    del sys.modules[modul]
print("File .env siap untuk peserta", os.environ.get("PARTICIPANT_ID"))

In [ ]:
# Persiapan 3 - Memeriksa koneksi ke model
from migas_shared.llm import chat

try:
    print("Jawaban model:", chat("Balas hanya dengan kata: SIAP").strip())
except Exception as error:  # noqa: BLE001
    raise RuntimeError("Model tidak menjawab. Periksa API key (jalankan ulang Persiapan 2) dan koneksi internet. "
                       f"Pesan asli: {error}") from error

Isi bagian bertanda `TODO(...)` dengan kode dari bagian **Kode Jawaban Latihan** pada PDF bab ini, lalu jalankan sel satu per satu
dengan Shift+Enter. Baris `... = None  # TODO` atau `raise NotImplementedError` berarti kode itu harus Anda ganti.

Mengubah catatan lapangan yang berantakan menjadi `LaporanInsiden` nested yang tervalidasi.

In [ ]:
# Langkah 1 - Menyiapkan model
import os
from typing import Literal, Optional

from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain_core.messages import HumanMessage
from langchain_openai import ChatOpenAI
from pydantic import BaseModel, Field

from migas_shared.data import KlasifikasiInsiden

load_dotenv()

model = ChatOpenAI(
    model=os.getenv("ONLINE_MODEL"),
    base_url=os.getenv("ONLINE_BASE_URL"),
    api_key=os.getenv("ONLINE_API_KEY"),
    reasoning_effort=os.getenv("LLM_REASONING_EFFORT", "none"),
    temperature=0,
)

## Langkah 2 - Skema laporan insiden
`LaporanInsiden` memuat daftar `Korban` dan klasifikasi `KlasifikasiInsiden`.

In [ ]:
class Korban(BaseModel):
    """Seseorang yang terlibat atau terdampak insiden."""
    # TODO(B4.1): definisikan field Korban: peran (str) dan cedera (Literal Tidak ada, Pertolongan pertama, Perawatan medis, Hilang hari kerja)
    pass  # TODO: definisikan field


class LaporanInsiden(BaseModel):
    """Laporan insiden terstruktur dari catatan lapangan."""
    # TODO(B4.2): definisikan field LaporanInsiden: lokasi, jam_kejadian (opsional), uraian (maksimal 200 karakter), korban (daftar Korban), tindakan_segera (daftar string), dan klasifikasi (KlasifikasiInsiden)
    pass  # TODO: definisikan field


# TODO(B4.3): buat agen tanpa tool dengan system prompt yang melarang menebak jam atau korban, dan response_format=LaporanInsiden
agent = None  # TODO: ganti dengan implementasi Anda

## Langkah 3 - Catatan 1: baut jatuh tanpa korban

In [ ]:
teks = (
    "kmrn sore sekitar jam 4an di GS-SR1 ada baut jatuh dr pipe rack pas lifting. ga ada yg luka. "
    "area dibarikade, kerjaan distop dulu, spv udah dilapor."
)
print(f"Catatan : {teks}")
response = agent.invoke({"messages": [HumanMessage(content=teks)]})
# TODO(B4.4): ambil hasil terstruktur dari response (kunci structured_response)
laporan = None  # TODO: ganti dengan implementasi Anda
if laporan is None:
    print("   Tidak ada hasil terstruktur\n")
else:
    print(laporan.model_dump_json(indent=2))
    print()

## Langkah 4 - Catatan 2: hose pecah dengan korban

In [ ]:
teks = (
    "Pad sumur KN-061 pagi 09.40, hose hidrolik pecah, oli panas kena tangan operator (Pak Dedi). "
    "Dikasih P3K lalu dibawa ke klinik lapangan, dirawat dan boleh kembali bertugas. Area dibersihkan pakai absorbent."
)
print(f"Catatan : {teks}")
response = agent.invoke({"messages": [HumanMessage(content=teks)]})
laporan = response.get("structured_response")
if laporan is None:
    print("   Tidak ada hasil terstruktur\n")
else:
    print(laporan.model_dump_json(indent=2))
    print()